In [1]:
import multiprocessing

In [4]:
!pip show multiprocessing

In [ ]:
import glob
import os
import re
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.utils import to_categorical

In [ ]:
from kmeans_multiprocessing import KMeansMultiprocessing
from kmeans_multiprocessing import KMeansMPWrapper
import mlflow.pyfunc
import mlflow
import numpy as np
from multiprocessing import Pool, cpu_count, shared_memory
import os
os.environ['MLFLOW_ENABLE_MODEL_REGISTRY'] = 'false'
import ctypes
import time

# Start MLflow run
with mlflow.start_run(run_name="mult12"):
    # Setup Parsl configuration
    # Disable all parsl logging before loading
    import logging
    # MLflow: log parameters
    mlflow.log_param("n_clusters", 3)
    mlflow.log_param("max_iter", 300)
    mlflow.log_param("num_workers", 12)
    mlflow.log_param("num_folds", 10)
    mlflow.log_param("num_images", len(images_flat))
    num_features = images_flat.shape[1]
    mlflow.log_param("num_features", num_features)
    mult_time =[]
    for i in range(10):
        debut = time.time()
        kmeans_mult = KMeansMultiprocessing(n_clusters= 3, max_iter=300, n_workers=12, use_shared_memory=True)
        kmeans_mult.fit(images_flat)
        fin = time.time()
        mult_time.append(fin-debut)

    training_time = np.mean(mult_time)
    print(f"Training time = {training_time:.2f} seconds")

    # MLflow: log training time
    mlflow.log_metric("avg training time", training_time)

    # Retrieve results from your KMeans class
    centroids = kmeans_mult.cluster_centers_
    num_iterations = kmeans_mult.number_of_iter
    cluster_labels = kmeans_mult.labels_

    # MLflow: Log iteration count
    mlflow.log_metric("iterations", num_iterations)

    # Compute cluster sizes
    unique, counts = np.unique(cluster_labels, return_counts=True)
    cluster_size_dict = {int(k): int(v) for k, v in zip(unique, counts)}

    # MLflow: Log size of each cluster
    for cluster_id, size in cluster_size_dict.items():
        mlflow.log_metric(f"cluster_{cluster_id}_size", size)
    

    print("\n✓ Logged centroids, iterations, and cluster sizes to MLflow successfully.\n")
    
